# 03 — MMR lambda sweep, adaptive controller, and uncertainty

**Goal:** compare a fixed relevance–diversity weight with the proposed query-adaptive controller. This notebook must use validation queries for design. The held-out test set is for the final locked evaluation only.

The controller uses two observable features: a normalized top-score gap as a query-confidence proxy, and average candidate cosine similarity as a shortlist-redundancy proxy. It maps these to a per-query λ. This is deliberately simple so its behavior can be inspected; whether it improves the Pareto frontier is an empirical question.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display, Image

ROOT = Path.cwd()
if not (ROOT / "src" / "image_rerank").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from image_rerank.experiment import load_cached_split, sweep_mmr_lambdas, evaluate_queries
from image_rerank.statistics import paired_bootstrap_difference
from image_rerank.plots import plot_lambda_sweep, plot_tradeoff
from image_rerank.artifacts import ensure_dir, write_dataframe, write_json

SPLIT = "validation"
ENCODER_KEY = "openclip-ViT-B-32-openai"
MAX_QUERIES = 3   # Set None for a full validation analysis.
OUTPUT_DIR = ensure_dir(ROOT / "artifacts" / "experiments" / "adaptive_controller")

queries = load_cached_split(ROOT / "artifacts" / "cache", SPLIT, ENCODER_KEY)
if MAX_QUERIES is not None:
    queries = queries[:MAX_QUERIES]
print("Validation query count:", len(queries))

## 1. Sweep fixed MMR weights

λ=1 is the encoder-similarity ranking (no redundancy penalty). Lower λ places more weight on novelty. The best fixed weight is selected from validation data, not test data.

In [ ]:
weights = np.round(np.arange(0.1, 1.01, 0.1), 2)
sweep_df = sweep_mmr_lambdas(queries, lambdas=weights, k=50)
write_dataframe(OUTPUT_DIR / "metrics" / "mmr_lambda_sweep_per_query.csv", sweep_df)

sweep_summary = sweep_df.groupby("lambda_relevance", as_index=False).agg(
    **{
        "ap@50_mean": ("ap@50", "mean"),
        "ap@50_std": ("ap@50", "std"),
        "ild@10_mean": ("ild@10", "mean"),
        "rerank_latency_ms_mean": ("rerank_latency_ms", "mean"),
    }
)
display(sweep_summary.round(4))

In [ ]:
lambda_plot_path = plot_lambda_sweep(
    sweep_df, OUTPUT_DIR / "figures" / "mmr_lambda_sweep.png"
)
display(Image(filename=str(lambda_plot_path)))

## 2. Compare adaptive MMR with the fixed λ=0.70 baseline

The paired bootstrap resamples queries, preserving the fact that each method sees the same shortlist. The confidence interval describes uncertainty in the mean per-query difference. A tiny smoke subset is not suitable for a publication claim; use the full validation split and report the number of queries.

In [ ]:
comparison_metrics, comparison_summary = evaluate_queries(
    queries,
    top_k=50,
    diversity_k=10,
    output_dir=OUTPUT_DIR,
    run_name="validation_controller_comparison",
)

ci_ap = paired_bootstrap_difference(
    comparison_metrics,
    metric="ap@50",
    treatment="adaptive_mmr",
    control="mmr_fixed_0.70",
    n_bootstrap=5000,
    seed=42,
)
ci_ild = paired_bootstrap_difference(
    comparison_metrics,
    metric="ild@10",
    treatment="adaptive_mmr",
    control="mmr_fixed_0.70",
    n_bootstrap=5000,
    seed=42,
)
print("AP@50 difference (adaptive - fixed):", ci_ap)
print("ILD@10 difference (adaptive - fixed):", ci_ild)
write_json(OUTPUT_DIR / "metrics" / "adaptive_vs_fixed_bootstrap_ci.json", {
    "ap@50": ci_ap,
    "ild@10": ci_ild,
    "warning": "Smoke subsets provide only a pipeline check; use full validation for inference.",
})
display(comparison_summary.round(4))

## 3. Controller diagnostics

`adaptive_mmr` rows include `lambda_relevance`, `query_confidence`, `candidate_redundancy`, and `top_score_gap_z`. Plotting λ against these diagnostics helps check that the controller behaves as intended rather than behaving like a near-constant weight.

In [ ]:
adaptive_rows = comparison_metrics.loc[
    comparison_metrics["method"] == "adaptive_mmr",
    ["query_id", "query", "lambda_relevance", "query_confidence", "candidate_redundancy", "top_score_gap_z", "ap@50", "ild@10"],
]
display(adaptive_rows.round(4))

import seaborn as sns
import matplotlib.pyplot as plt
sns.set_theme(style="darkgrid", context="notebook", palette="deep")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(data=adaptive_rows, x="query_confidence", y="lambda_relevance", ax=axes[0])
axes[0].set_title("Controller weight vs. score-gap confidence")
sns.scatterplot(data=adaptive_rows, x="candidate_redundancy", y="lambda_relevance", ax=axes[1])
axes[1].set_title("Controller weight vs. candidate redundancy")
fig.tight_layout()
controller_plot = OUTPUT_DIR / "figures" / "controller_diagnostics.png"
fig.savefig(controller_plot, bbox_inches="tight", dpi=180)
plt.close(fig)
display(Image(filename=str(controller_plot)))

## Decision rule for the next iteration

Do not call the adaptive controller better because its ILD is higher. Prefer it only if it improves the validation Pareto frontier under the preregistered relevance-loss and latency limits, with paired uncertainty estimates. If it does not, report the useful finding: a fixed λ or DPP is preferable for this candidate distribution.